# 🖥️ Backend Engineering RAG Tutor

A **Retrieval-Augmented Generation (RAG)** chatbot that answers backend engineering questions using your own PDF documents as a knowledge base.

## Architecture

```
PDFs → Chunking → Embeddings → ChromaDB Vector Store
                                        ↓
User Question → Retriever → Context → LLM → Answer
```

## Setup

### 1. Install dependencies
```bash
pip install langchain langchain-community langchain-huggingface langchain-openai
pip install chromadb gradio pypdf sentence-transformers
```

### 2. Set your OpenRouter API key
```bash
export OPENROUTER_API_KEY="your-key-here"
```

### 3. Add your PDFs
Place PDF files in a `docs/` folder next to this notebook:
```
docs/
  spring_boot.pdf
  microservices.pdf
  system_design.pdf
```


In [26]:
import os
from pathlib import Path

from langchain_chroma import Chroma

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

import gradio as gr

# -----------------------------
# OpenRouter LLM configuration
# -----------------------------
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),  # make sure this env variable is set
    model="openai/gpt-4o-mini",
    temperature=0.2
)

## 2. Document Loading

Scan the `docs/` folder and load every PDF. Each page becomes a separate LangChain `Document` object with metadata (source filename, page number) attached.

In [22]:

docs = []

for pdf in Path("docs").glob("*.pdf"):
    loader = PyPDFLoader(str(pdf))
    docs.extend(loader.load())

print(f"Loaded {len(docs)} documents")

Loaded 997 documents


## 3. Text Chunking

Large documents are split into smaller, overlapping chunks before indexing. This improves retrieval precision — the model finds the *exact paragraph* that answers a question rather than a whole chapter.

| Parameter | Value | Effect |
|---|---|---|
| `chunk_size` | 800 chars | Max length of each chunk |
| `chunk_overlap` | 150 chars | Overlap between adjacent chunks — prevents answers being cut off at boundaries |

> **Tuning tip:** For dense technical docs, 600–1000 chars with 100–200 overlap works well.

In [23]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

splits = splitter.split_documents(docs)

print(f"Split into {len(splits)} chunks")

Split into 4045 chunks


## 4. Embeddings + Vector Store (ChromaDB)

Each text chunk is converted into a dense vector (embedding) that captures its semantic meaning. These vectors are stored in **ChromaDB** — a local vector database — enabling fast similarity search at query time.

**Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`
- Lightweight (22M params), runs fully locally — no API calls for indexing
- Strong semantic similarity performance for technical text

**ChromaDB** persists to `./chroma_db/` on disk, so you only build the index once.  
> To force a full rebuild, delete the `chroma_db/` folder and re-run this cell.

In [28]:
DB_NAME = "vector_db"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    persist_directory=DB_NAME,
    embedding_function=embedding_model
)

vectorstore.add_documents(splits)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Vector store created and retriever ready")

Vector store created and retriever ready


## 5. RAG Pipeline

The core RAG loop runs in four steps for every question:

1. **Embed** the question using the same model used during indexing
2. **Retrieve** the 4 most semantically similar chunks from ChromaDB
3. **Build a prompt** injecting those chunks as grounding context
4. **Call the LLM** and return the answer + source filenames

> The prompt instructs the model to answer using *only* the provided context. This prevents hallucination outside your documents.

In [29]:
def rag_answer(question):

    # Retrieve relevant chunks
    docs = retriever.invoke(question)
    
    # Join chunks as context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Prompt
    prompt = f"""
You are an expert backend engineering tutor.

Answer the question using ONLY the context provided below.

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)

    # Get sources
    sources = [doc.metadata.get("source") for doc in docs]

    return response.content, sources

## 6. Gradio Chat Interface

A simple chat UI built with **Gradio**. The `respond` function:
- Calls `rag_answer` with the latest user message
- Appends source references (as a markdown list) to the answer
- Updates the conversation history for multi-turn display

Run the next cell and open the local URL that appears (typically `http://127.0.0.1:7860`).

In [30]:
def respond(message, history):

    answer, sources = rag_answer(message)

    # Add sources for reference
    answer += "\n\nSources:\n" + "\n".join(set(sources))

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})

    return history

with gr.Blocks() as demo:

    gr.Markdown("# Backend Engineering Tutor (RAG)")

    chatbot = gr.Chatbot(type="messages")

    msg = gr.Textbox(placeholder="Ask a backend/Spring question...")

    msg.submit(respond, [msg, chatbot], chatbot)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
